# Lazy Iteration, Generators, and `yield` (with Tick Data Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to:

- **Lazy iteration** in Python
- **Generator functions** and the `yield` keyword
- **Generator expressions** and iterator pipelines
- Practical patterns for **large data processing**

We will use trading-/quant-style examples throughout, especially:

- Streaming **tick data** from files
- Iterating lazily over large **TimescaleDB** (or similar) query results
- Building **processing pipelines** (filtering, mapping, windowing) without loading
  everything into RAM.

We will build up gradually:

1. **Core ideas**: iterables, iterators, and simple `yield`
2. **Generator functions** vs normal functions
3. **Generator expressions** and lazy pipelines
4. **Tick file reading line by line**
5. **Simulated TimescaleDB tick streaming** and backpressure
6. **Composing generators for data cleaning, filtering, and aggregation**
7. **Advanced patterns**: `send`, `yield from`, and resource management
8. **Best practices and common pitfalls**

> Run and modify the code cells as you go. The goal is to make generators feel like a
> natural tool for *scaling up* your tick / time-series processing logic.

### Contents

- [1. Core Ideas: Iterables, Iterators, and yield](#1-core-ideas-iterables-iterators-and-yield)
- [2. Generator Functions: Basic Patterns](#2-generator-functions-basic-patterns)
- [3. Generator Expressions and Lazy Pipelines](#3-generator-expressions-and-lazy-pipelines)
- [4. Tick File Reading Line by Line](#4-tick-file-reading-line-by-line)
- [5. Simulated TimescaleDB Streaming of Tick Data](#5-simulated-timescaledb-streaming-of-tick-data)
- [6. Advanced Generator Features: yield from and send](#6-advanced-generator-features-yield-from-and-send)
- [7. Best Practices and Common Pitfalls](#7-best-practices-and-common-pitfalls)

## 1. Core Ideas: Iterables, Iterators, and `yield`

Before using generators fluently, be clear on three concepts:

- **Iterator**: an object that produces values **on demand** via `__next__`.
- **Iterable**: an object you can loop over (e.g. list, tuple, file, range).
<br>Consider it as a subset of iterators: all iterables are iterators.
- **Generator**: a special kind of iterator created by a **generator function** or
  **generator expression**.

The key property for performance:

> Generators **do not compute all values up front**; they compute each value
> **only when requested**, which is ideal for large tick streams.

In [ ]:
# 1.1 Simple iterator vs generator

# A list is an iterable; iter(list) gives you an iterator
prices = [100.0, 100.5, 101.2]

it = iter(prices)
print(next(it))  # first price
print(next(it))  # second price
print(next(it))  # third price

# next(it) now would raise StopIteration

In [ ]:
# 1.2 A minimal generator function

# Any function that uses `yield` becomes a generator function.

def simple_price_generator():
    print("Generating first price")
    yield 100.0  # pause here
    print("Generating second price")
    yield 101.0  # pause here
    print("Generating third price")
    yield 102.0  # pause here


gen = simple_price_generator()  # no code runs yet

print("Calling next(gen):", next(gen))  # runs until first yield
print("Calling next(gen):", next(gen))  # resumes after previous yield
print("Calling next(gen):", next(gen))  # resumes again

try:
    next(gen)  # generator is exhausted, raises StopIteration
except StopIteration:
    print("Generator exhausted")

### 1.3 Why generators matter for tick data

Imagine a TimescaleDB hypertable with **hundreds of millions of ticks**.
You do *not* want to:

- Fetch all rows into memory as a giant list.
- Deserialize everything up front.

Instead, you want to **stream rows one by one** (or in chunks), process them, and
optionally **stop early** if you already have what you need.

Generators are Python's **native tool** to express this style of lazy, streaming
computation.

## 2. Generator Functions: Basic Patterns

We'll start with a few basic patterns, then map them to tick data.

### 2.1 Range-like generator

A classic pattern is a generator that behaves like `range`, but you can embed
**custom logic** inside.

In [92]:
# 2.1 Range-like price generator

import numpy
from collections.abc import Iterator

def price_range(start: float, change: float, steps: int) -> Iterator[float]:
    """Generate prices from start to stop (exclusive) with given step."""
    price = start
    for step in range(steps + 1):
        yield price  # lazily emit next price
        price += change * numpy.random.randint(-1, 2)


for p in price_range(1234.56, 0.01, 50):
    print(p, end = ", ")

1234.56, 1234.57, 1234.57, 1234.58, 1234.59, 1234.59, 1234.6, 1234.59, 1234.6, 1234.61, 1234.6, 1234.6, 1234.6, 1234.61, 1234.61, 1234.6, 1234.61, 1234.62, 1234.6299999999999, 1234.62, 1234.6299999999999, 1234.62, 1234.61, 1234.6, 1234.61, 1234.6, 1234.59, 1234.58, 1234.57, 1234.58, 1234.58, 1234.59, 1234.59, 1234.58, 1234.58, 1234.57, 1234.58, 1234.57, 1234.56, 1234.56, 1234.57, 1234.57, 1234.57, 1234.58, 1234.57, 1234.56, 1234.55, 1234.56, 1234.56, 1234.57, 1234.57, 

### 2.2 Filtering and mapping with generators

A key idea: **generators can be chained**. Each stage consumes one item at a time and
passes it on.

We'll simulate a simple price stream, then build generators that:

- Filter only certain symbols.
- Convert raw ticks into mid-prices (for bid/ask).

We'll use a simple in-memory list first; later we'll swap the source for a file or DB cursor.

In [ ]:
# 2.2 Chaining generator functions: filter + map

from dataclasses import dataclass
from datetime import datetime
from typing import Iterable


@dataclass(slots=True)
class RawTick:
    symbol: str
    bid: float
    ask: float
    ts: datetime


def iter_ticks(source: Iterable[RawTick]):
    """Base generator: just re-yield ticks from any iterable.

    Later we can plug in a DB cursor or file iterator here.
    """
    for tick in source:
        yield tick


def filter_symbol(ticks: Iterable[RawTick], symbol: str):
    """Yield only ticks for a given symbol."""
    for tick in ticks:
        if tick.symbol == symbol:
            yield tick


@dataclass(slots=True)
class MidPrice:
    symbol: str
    mid: float
    ts: datetime


def to_midprice(ticks: Iterable[RawTick]):
    """Convert raw bid/ask ticks into mid-price objects."""
    for tick in ticks:
        mid = (tick.bid + tick.ask) / 2
        yield MidPrice(symbol=tick.symbol, mid=mid, ts=tick.ts)


# Demo with a small in-memory list of ticks
ticks = [
    RawTick("AAPL", bid=100.0, ask=100.1, ts=datetime.utcnow()),
    RawTick("MSFT", bid=200.0, ask=200.2, ts=datetime.utcnow()),
    RawTick("AAPL", bid=100.2, ask=100.3, ts=datetime.utcnow()),
]

pipeline = to_midprice(filter_symbol(iter_ticks(ticks), "AAPL"))

for mp in pipeline:
    print(mp)


## 3. Generator Expressions and Lazy Pipelines

Besides generator functions, Python has **generator expressions**:

```python
(expr for x in iterable if condition)
```

They are like list comprehensions but **lazy**.

### 3.1 Basic generator expression

We'll create a lazy mid-price stream from a list of prices.

In [ ]:
# 3.1 Simple generator expression

prices = [100.0, 100.2, 100.4, 100.6]

# Lazily square each price (just as a demo)
squares = (p * p for p in prices)

print(next(squares))
print(list(squares))  # remaining values

In [ ]:
# 3.2 Generator expression pipeline for ticks

from datetime import timedelta


# Reuse RawTick from earlier
now = datetime.utcnow()
raw_ticks = [
    RawTick("AAPL", 100.0, 100.1, now),
    RawTick("AAPL", 100.1, 100.2, now + timedelta(seconds=1)),
    RawTick("MSFT", 200.0, 200.1, now + timedelta(seconds=2)),
    RawTick("AAPL", 100.2, 100.3, now + timedelta(seconds=3)),
]

# Pipeline: filter AAPL, compute mid, keep only mid > 100.15
mid_pipeline = (
    MidPrice(t.symbol, (t.bid + t.ask) / 2, t.ts)
    for t in raw_ticks
    if t.symbol == "AAPL"
)

high_mids = (mp for mp in mid_pipeline if mp.mid > 100.15)

for mp in high_mids:
    print(mp)


Generator expressions are great for **inline, one-off pipelines**. For more complex
or reusable logic, prefer **named generator functions**, which are easier to test
and compose.

Next, we'll focus on **file-based tick reading**, then a more realistic
**TimescaleDB-style streaming** pattern.

## 4. Tick File Reading Line by Line

A classic I/O pattern with generators:

- Open a large **tick file** (e.g. CSV or TSV).
- Stream **one line at a time**, parse into a tick object, process, then discard.

We avoid reading the whole file into memory, which is critical for large historical
backtests or feature pipelines.

In [1]:
# 4.1 Generator to read ticks from a CSV-like file

from dataclasses import dataclass
from datetime import datetime
from typing import TextIO, Iterator


@dataclass(slots=True)
class CsvTick:
    symbol: str
    price: float
    size: float
    ts: datetime


def iter_csv_ticks(f: TextIO) -> Iterator[CsvTick]:
    """Yield CsvTick objects line by line from an open file.

    Expected columns: symbol,price,size,timestamp_iso
    """
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue  # skip blanks/comments
        symbol, price_s, size_s, ts_s = line.split(",")
        yield CsvTick(
            symbol=symbol,
            price=float(price_s),
            size=float(size_s),
            ts=datetime.fromisoformat(ts_s),
        )


# Demo: simulate a small CSV file using io.StringIO
import io

csv_data = """# symbol,price,size,timestamp
AAPL,100.0,10,2024-01-01T09:30:00
AAPL,100.1,15,2024-01-01T09:30:01
MSFT,200.0,5,2024-01-01T09:30:02
"""

f = io.StringIO(csv_data)

for tick in iter_csv_ticks(f):
    print(tick)


CsvTick(symbol='AAPL', price=100.0, size=10.0, ts=datetime.datetime(2024, 1, 1, 9, 30))
CsvTick(symbol='AAPL', price=100.1, size=15.0, ts=datetime.datetime(2024, 1, 1, 9, 30, 1))
CsvTick(symbol='MSFT', price=200.0, size=5.0, ts=datetime.datetime(2024, 1, 1, 9, 30, 2))


In [ ]:
# 4.2 Building a processing pipeline on top of file iteration

from typing import Iterable


def filter_symbol_csv(ticks: Iterable[CsvTick], symbol: str) -> Iterator[CsvTick]:
    for tick in ticks:
        if tick.symbol == symbol:
            yield tick


def vwap(ticks: Iterable[CsvTick]) -> float:
    total_notional = 0.0
    total_size = 0.0
    for tick in ticks:
        total_notional += tick.price * tick.size
        total_size += tick.size
    return total_notional / total_size if total_size else float("nan")


f = io.StringIO(csv_data)
print("VWAP for AAPL:", vwap(filter_symbol_csv(iter_csv_ticks(f), "AAPL")))

## 5. Simulated TimescaleDB Streaming of Tick Data

Real TimescaleDB / PostgreSQL drivers expose **cursor objects** that let you
fetch rows in chunks (e.g. `fetchmany`) or iterate row-by-row.

We will simulate this pattern:

- A `DbCursor` that yields rows lazily in **batches**.
- A generator wrapper that turns those rows into domain objects (`DbTick`).
- A processing pipeline on top of that stream.

This pattern generalizes to any DB that supports server-side cursors / streaming.

In [ ]:
# 5.1 Simulating a DB cursor that yields rows in batches

from dataclasses import dataclass
from typing import Sequence, Any, Iterator, Iterable


@dataclass(slots=True)
class DbTick:
    symbol: str
    price: float
    size: float
    ts: datetime


class FakeDbCursor:
    """Simulate a DB cursor over a large tick result set.

    In reality, you'd use something like psycopg's cursor with server-side cursors.
    """

    def __init__(self, rows: Sequence[tuple[Any, ...]], batch_size: int = 2):
        self._rows = rows
        self._batch_size = batch_size
        self._index = 0

    def fetchmany(self) -> list[tuple[Any, ...]]:
        # Fetch next batch of rows
        if self._index >= len(self._rows):
            return []
        batch = self._rows[self._index : self._index + self._batch_size]
        self._index += len(batch)
        return list(batch)


def iter_db_ticks(cursor: FakeDbCursor) -> Iterator[DbTick]:
    """Lazily yield DbTick objects from a DB cursor.

    This is the core streaming primitive over a large TimescaleDB query.
    """
    while True:
        batch = cursor.fetchmany()
        if not batch:
            break  # no more rows
        for symbol, price, size, ts in batch:
            yield DbTick(symbol=symbol, price=price, size=size, ts=ts)


# Demo: simulate 5 rows coming from the DB
rows = [
    ("AAPL", 100.0, 10.0, datetime(2024, 1, 1, 9, 30, 0)),
    ("AAPL", 100.1, 5.0, datetime(2024, 1, 1, 9, 30, 1)),
    ("MSFT", 200.0, 2.0, datetime(2024, 1, 1, 9, 30, 2)),
    ("AAPL", 100.2, 3.0, datetime(2024, 1, 1, 9, 30, 3)),
    ("AAPL", 100.3, 7.0, datetime(2024, 1, 1, 9, 30, 4)),
]

cursor = FakeDbCursor(rows, batch_size=2)

for t in iter_db_ticks(cursor):
    print(t)


In [ ]:
# 5.2 Processing a large DB tick stream lazily

from typing import Iterable


def filter_symbol_db(ticks: Iterable[DbTick], symbol: str) -> Iterator[DbTick]:
    for tick in ticks:
        if tick.symbol == symbol:
            yield tick


def rolling_vwap(ticks: Iterable[DbTick], window_size: int) -> Iterator[tuple[datetime, float]]:
    """Compute a simple rolling VWAP over the last N ticks.

    This keeps only O(window_size) data in memory.
    """
    from collections import deque

    window: deque[DbTick] = deque(maxlen=window_size)

    for tick in ticks:
        window.append(tick)
        total_notional = sum(t.price * t.size for t in window)
        total_size = sum(t.size for t in window)
        vwap_value = total_notional / total_size if total_size else float("nan")
        yield tick.ts, vwap_value


cursor = FakeDbCursor(rows, batch_size=2)
stream = iter_db_ticks(cursor)

for ts, v in rolling_vwap(filter_symbol_db(stream, "AAPL"), window_size=3):
    print(ts, "VWAP (last 3 AAPL ticks):", v)


This pattern scales naturally:

- Only a **small window** of data is in memory at any time.
- You can stop iterating early (e.g. after N points) without reading the whole result set.
- Swapping from a fake cursor to a real TimescaleDB cursor should be mostly a matter
  of changing `FakeDbCursor`.

Next, we'll look at some **advanced generator features** (`yield from`, `send`) and
how they can be useful (or overkill) in data-processing pipelines.

## 6. Advanced Generator Features: `yield from` and `send`

Most trading data pipelines only need **simple, one-way generators** (we pull values out).

Python also supports more advanced patterns:

- **`yield from subgen`**: delegate part of a generator's work to another generator.
- **`.send(value)`**: send data **into** a generator, turning it into a coroutine-like
  consumer (useful for sinks or stateful processors).

We'll briefly illustrate these so you recognize them in the wild.

In [ ]:
# 6.1 Using `yield from` to flatten nested generators

from collections.abc import Iterable


def iter_many_sources(sources: Iterable[Iterable[DbTick]]) -> Iterator[DbTick]:
    """Flatten multiple DbTick sources into a single stream.

    For example, many small TimescaleDB queries for different days.
    """
    for src in sources:
        # `yield from` delegates iteration to another iterable/generator
        yield from src


# Demo: pretend we queried two different date ranges
cursor1 = FakeDbCursor(rows[:3], batch_size=2)
cursor2 = FakeDbCursor(rows[3:], batch_size=2)

combined_stream = iter_many_sources([
    iter_db_ticks(cursor1),
    iter_db_ticks(cursor2),
])

for t in combined_stream:
    print(t)


In [ ]:
# 6.2 Using `send` to create a simple tick consumer

from typing import Optional


def vwap_consumer():
    """A generator that *consumes* ticks via .send and prints live VWAP.

    This is more of a teaching example; in real code you'd likely wrap this
    into a class or function, but it shows how `send` works.
    """
    total_notional = 0.0
    total_size = 0.0
    vwap_value: Optional[float] = None

    while True:
        tick = yield vwap_value  # wait for next tick to be sent in
        if tick is None:
            continue
        total_notional += tick.price * tick.size
        total_size += tick.size
        vwap_value = total_notional / total_size if total_size else float("nan")
        print(f"[vwap_consumer] New tick {tick.symbol} {tick.price} size {tick.size}, VWAP={vwap_value}")


# Demo: drive the consumer with a few DbTicks
consumer = vwap_consumer()
next(consumer)  # prime the generator (run up to first yield)

for t in iter_db_ticks(FakeDbCursor(rows, batch_size=2)):
    consumer.send(t)

consumer.close()  # tidy up


In modern Python, more complex coroutine-style logic is often written using
`async`/`await` rather than bare generator `.send()`. Still, you will see
this pattern in libraries and it's useful to understand conceptually.

For most data pipelines, **plain `for` loops over generator functions** are
sufficient and clearer.

## 7. Best Practices and Common Pitfalls

### 7.1 Good patterns for generators in trading/time-series code

- **Wrap external sources** (files, DB cursors, sockets) in small generator functions
  that yield domain objects (`Tick`, `DbTick`, etc.).
- Build **small, composable stages**: filter, map, aggregate, window.
- Keep each stage **stateless or minimally stateful**, and document side effects.
- Use `itertools` helpers (`islice`, `takewhile`, `groupby`, etc.) to enrich pipelines.

### 7.2 Avoid these pitfalls

- **Exhausting a generator twice**: once consumed, it's done. Re-create if needed.
- **Accidentally materializing**: calling `list(gen)` brings everything into memory.
  Do that only at the edges (e.g. tests, small results).
- **Complex nested generators** that are hard to read: prefer named functions over
  very long generator expressions.

### 7.3 How to practice

- Replace simple list-based processing in your own tick backtest scripts with
  generator-based pipelines.
- Wrap a real TimescaleDB query cursor in a generator like `iter_db_ticks` and
  add stages for filtering, windowing, and feature computation.
- Experiment with `itertools` (`islice`, `accumulate`, `pairwise`) inside generator
  functions to build more advanced indicators.

When these patterns feel natural, you will be able to scale your tick/time-series
processing with **minimal memory footprint** and **clean, composable code**.